# Qualtran QSVT Builder From Block-Encoding

This notebook builds a QSVT-style sequence using Qualtran bloqs.

Phase modes:
- `single_qubit`: standard QSVT phase `Rz(2*phi)` on a 1-qubit signal register.
- `projector_zero`: phase on `|0...0>` for a multi-qubit signal register.

`projector_zero` supports block-encodings whose postselection subspace is `|00...0>`.

In [11]:
import numpy as np

from qualtran import Bloq, BloqBuilder, CompositeBloq, Controlled, CtrlSpec, QAny, QUInt
from qualtran.bloqs.arithmetic import XorK
from qualtran.bloqs.basic_gates import CNOT, CRz, GlobalPhase, Rz
from qualtran.resource_counting import QECGatesCost, get_cost_value


In [12]:
def make_query_schedule(query_bloq: Bloq, n_queries: int, alternate_adjoint: bool = True) -> list[Bloq]:
    schedule = []
    for k in range(n_queries):
        schedule.append(query_bloq.adjoint() if (alternate_adjoint and (k % 2 == 1)) else query_bloq)
    return schedule


def _apply_phase_single_qubit(bb: BloqBuilder, state: dict[str, object], signal_reg: str, phi: float):
    signal = state[signal_reg]
    # In this mode, signal must be exactly 1 qubit.
    out = bb.add_d(Rz(angle=2.0 * float(phi)), q=signal)
    state[signal_reg] = out['q']


def _apply_phase_projector_zero(bb: BloqBuilder, state: dict[str, object], signal_reg: str, phi: float, signal_bitsize: int):
    """Apply exp(i*phi*|0...0><0...0|) on packed signal register."""
    if signal_bitsize < 1:
        raise ValueError('signal_bitsize must be >= 1.')

    sig = state[signal_reg]
    all_ones = (1 << signal_bitsize) - 1

    # Map |0...0> <-> |1...1> using packed XOR mask.
    s1 = bb.add_d(XorK(QUInt(signal_bitsize), all_ones), x=sig)['x']

    # Apply e^{i phi} only when ctrl register is |1...1>.
    ctrl_spec = CtrlSpec(qdtypes=QAny(signal_bitsize), cvs=all_ones)
    cond_phase = Controlled(GlobalPhase(exponent=float(phi) / np.pi), ctrl_spec=ctrl_spec)
    s2 = bb.add_d(cond_phase, ctrl=s1)['ctrl']

    # Undo mapping.
    s3 = bb.add_d(XorK(QUInt(signal_bitsize), all_ones), x=s2)['x']
    state[signal_reg] = s3


def _apply_phase(
    bb: BloqBuilder,
    state: dict[str, object],
    signal_reg: str,
    phi: float,
    phase_mode: str,
    signal_bitsize: int,
):
    if phase_mode == 'single_qubit':
        if signal_bitsize != 1:
            raise ValueError(
                'single_qubit phase mode requires signal_bitsize == 1. '
                f'Got signal_bitsize={signal_bitsize}. Use phase_mode=\'projector_zero\'.'
            )
        _apply_phase_single_qubit(bb, state, signal_reg, phi)
    elif phase_mode == 'projector_zero':
        _apply_phase_projector_zero(bb, state, signal_reg, phi, signal_bitsize)
    else:
        raise ValueError("phase_mode must be 'single_qubit' or 'projector_zero'.")


def _apply_query(bb: BloqBuilder, state: dict[str, object], query_bloq: Bloq):
    in_names = [reg.name for reg in query_bloq.signature.lefts()]
    missing = [name for name in in_names if name not in state]
    if missing:
        raise ValueError(f"State is missing query registers: {missing}")
    out = bb.add_d(query_bloq, **{name: state[name] for name in in_names})
    state.update(out)


def build_qsvt_composite(
    query_schedule: list[Bloq],
    phases: list[float] | np.ndarray,
    register_bitsizes: dict[str, int],
    signal_reg: str = 'signal',
    phase_mode: str = 'single_qubit',
) -> CompositeBloq:
    phases = list(phases)
    if len(phases) != len(query_schedule) + 1:
        raise ValueError(
            f"Need len(phases)=len(query_schedule)+1, got {len(phases)} and {len(query_schedule)}"
        )
    if signal_reg not in register_bitsizes:
        raise ValueError(f"signal_reg '{signal_reg}' not in register_bitsizes")

    signal_bitsize = int(register_bitsizes[signal_reg])

    bb = BloqBuilder()
    state: dict[str, object] = {}
    for name, bitsize in register_bitsizes.items():
        state[name] = bb.add_register(name, bitsize)

    _apply_phase(bb, state, signal_reg, phases[0], phase_mode, signal_bitsize)
    for k, query in enumerate(query_schedule):
        _apply_query(bb, state, query)
        _apply_phase(bb, state, signal_reg, phases[k + 1], phase_mode, signal_bitsize)

    return bb.finalize(**state)


In [13]:
# Demo 1: standard single-qubit signal mode
bb_u = BloqBuilder()
signal = bb_u.add_register('signal', 1)
system = bb_u.add_register('system', 1)
o1 = bb_u.add_d(CRz(angle=0.7), ctrl=signal, q=system)
o2 = bb_u.add_d(CNOT(), ctrl=o1['q'], target=o1['ctrl'])
U_block = bb_u.finalize(signal=o2['target'], system=o2['ctrl'])

degree = 6
phases = np.linspace(0.0, np.pi / 2, degree + 1)
schedule = make_query_schedule(U_block, n_queries=degree, alternate_adjoint=True)

qsvt_single = build_qsvt_composite(
    query_schedule=schedule,
    phases=phases,
    register_bitsizes={'signal': 1, 'system': 1},
    signal_reg='signal',
    phase_mode='single_qubit',
)

print('single_qubit counts:', get_cost_value(qsvt_single, QECGatesCost()).asdict())


single_qubit counts: {'clifford': 21, 'rotation': 16}


In [14]:
# Demo 2: multi-qubit signal mode (projector onto |00...0>)
bb_m = BloqBuilder()
sig2 = bb_m.add_register('signal', 2)
sys1 = bb_m.add_register('system', 1)
sig2 = bb_m.add_d(XorK(QUInt(2), 1), x=sig2)['x']
sys1 = bb_m.add_d(Rz(angle=0.25), q=sys1)['q']
U_multi_signal = bb_m.finalize(signal=sig2, system=sys1)

degree2 = 4
phases2 = [0.12, -0.31, 0.20, -0.41, 0.15]
schedule2 = make_query_schedule(U_multi_signal, n_queries=degree2, alternate_adjoint=True)

qsvt_multi = build_qsvt_composite(
    query_schedule=schedule2,
    phases=phases2,
    register_bitsizes={'signal': 2, 'system': 1},
    signal_reg='signal',
    phase_mode='projector_zero',
)

print('projector_zero QSVT built successfully (cost key may be unavailable for Controlled(GlobalPhase) in this Qualtran version).')


projector_zero QSVT built successfully (cost key may be unavailable for Controlled(GlobalPhase) in this Qualtran version).


In [15]:
# Optional Cirq conversion
c1 = qsvt_single.to_cirq_circuit()
c2 = qsvt_multi.to_cirq_circuit()
print('single mode ops:', len(list(c1.all_operations())))
print('multi mode ops :', len(list(c2.all_operations())))


single mode ops: 13
multi mode ops : 19
